# This is where we can train our own model

In [1]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

Load the data for training

In [2]:
df = pd.read_csv("../data/train_images.csv")
attributes = np.load("../data/attributes.npy", allow_pickle=True)
attributes = (attributes - attributes.min(axis=1, keepdims=True)) / (
    attributes.max(axis=1, keepdims=True) - attributes.min(axis=1, keepdims=True)
)

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Replace with your data
class_names = [i + 1 for i in range(200)]

scaler = StandardScaler()
attrs_scaled = scaler.fit_transform(attributes)

pca = PCA(n_components=65)
pca_2d = pca.fit_transform(attrs_scaled)

print(f'Cumulative variance: {np.sum(pca.explained_variance_ratio_):.1%}')

Cumulative variance: 91.9%


In [4]:
class ImageAttributeDataset(Dataset):
    def __init__(self, data_df, attrs, transform):
        self.data = data_df
        self.transform = transform
        self.attributes = attrs

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = "../data" + self.data.iloc[idx]['image_path']
        label = self.data.iloc[idx]['label']
        # subtract 1 from label to go to attribute index
        attrs = torch.tensor(self.attributes[label - 1], dtype=torch.float32)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, attrs


In [5]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

In [6]:
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


train_dataset = ImageAttributeDataset(train_df, transform=train_tfms, attrs=attributes)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [7]:
classes = train_df['label'].unique()

Train the model

In [8]:
class ImageEmbeddingModel(nn.Module):
    def __init__(self, embed_dim=312, img_size=224):
        super().__init__()
        self.img_size = img_size
        x = torch.randn(1, 3, img_size, img_size)

        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 27, padding=13),   # Output: 32x224x224
            nn.ReLU(),
            nn.MaxPool2d(2),                    # Output: 32x112x112

            nn.Conv2d(32, 64, 27, padding=13),  # Output: 64x112x112
            nn.ReLU(),
            nn.MaxPool2d(2),                    # Output: 64x56x56

            nn.Conv2d(64, 128, 27, padding=13), # Output: 128x56x56
            nn.ReLU(),
            nn.MaxPool2d(2),                    # Output: 128x28x28
            #
            # nn.Conv2d(128, 256, 27, padding=13),# Output: 256x28x28
            # nn.ReLU(),
            # nn.MaxPool2d(2)                     # Output: 256x14x14
        )

        # Dynamic size computation (works for any img_size)
        with torch.no_grad():
            x = self.backbone(x)
            flat_size = x.numel() // x.shape[0]  # 50176 for 224x224

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, embed_dim)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ImageEmbeddingModel().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
criterion = nn.CosineEmbeddingLoss()

# Training
model.train()
for epoch in range(10):  # Adjust epochs
    total_loss = 0
    for images, attrs in train_loader:

        images, attrs = images.to(device), attrs.to(device)
        optimizer.zero_grad()
        preds = model(images)
        target = torch.ones(preds.size(0)).to(device)
        loss = criterion(preds, attrs, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}')

print("Finished training")


Run the model on the val dataset

In [109]:
val_dataset = ImageAttributeDataset(val_df, transform=test_tfms, attrs=attributes)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

val_loss = 0
criterion = nn.CosineEmbeddingLoss()
predictions = []
with torch.no_grad():
    for images, attrs in val_loader:
        images, attrs = images.to(device), attrs.to(device)
        preds = model(images)
        predictions.append(preds.tolist())
        loss = criterion(preds, attrs)
        val_loss += loss.item()
val_loss /= len(val_loader)
print(f'Validation MSE Loss: {val_loss:.4f}')

TypeError: CosineEmbeddingLoss.forward() missing 1 required positional argument: 'target'

In [85]:
def predict_class(embedding, class_embeddings):
    # embedding: tensor of shape (embedding_dim,)
    # class_embeddings: tensor of shape (num_classes, embedding_dim)
    # class_names: list of class labels

    embedding = embedding.unsqueeze(0)  # shape (1, embedding_dim)
    similarities = F.cosine_similarity(embedding, class_embeddings)  # shape (num_classes,)
    best_idx = torch.argmax(similarities).item()
    return best_idx + 1

In [86]:
val_df["embeddings"] = predictions

In [87]:
val_df

,image_path,label,embeddings
2004,/train_images/2005.jpg,70,"[0.0055096750147640705, 0.0292478259652853, 0...."
1711,/train_images/1712.jpg,59,"[0.006831617560237646, 0.028513098135590553, 0..."
2252,/train_images/2253.jpg,81,"[0.005965130869299173, 0.02135973982512951, 0...."
747,/train_images/748.jpg,25,"[0.004733365494757891, 0.026003194972872734, 0..."
2346,/train_images/2347.jpg,85,"[0.005643811076879501, 0.02558925934135914, 0...."
...,...,...,...
1327,/train_images/1328.jpg,45,"[0.008209949359297752, 0.03717655688524246, 0...."
2859,/train_images/2860.jpg,110,"[0.006007458083331585, 0.02774239517748356, 0...."
23,/train_images/24.jpg,1,"[0.00618132296949625, 0.02886546589434147, 0.0..."
3071,/train_images/3072.jpg,122,"[0.007186871953308582, 0.028653042390942574, 0..."


In [88]:
predicted_labels = []
for idx, row in val_df.iterrows():
    predicted_class = predict_class(
        torch.tensor(row["embeddings"]),
        torch.tensor(attributes)
    )
    predicted_labels.append(predicted_class)

In [92]:
val_df['predicted_label'] = predicted_labels

In [93]:
val_df

,image_path,label,embeddings,predicted_label
2004,/train_images/2005.jpg,70,"[0.0055096750147640705, 0.0292478259652853, 0....",25
1711,/train_images/1712.jpg,59,"[0.006831617560237646, 0.028513098135590553, 0...",25
2252,/train_images/2253.jpg,81,"[0.005965130869299173, 0.02135973982512951, 0....",25
747,/train_images/748.jpg,25,"[0.004733365494757891, 0.026003194972872734, 0...",25
2346,/train_images/2347.jpg,85,"[0.005643811076879501, 0.02558925934135914, 0....",25
...,...,...,...,...
1327,/train_images/1328.jpg,45,"[0.008209949359297752, 0.03717655688524246, 0....",25
2859,/train_images/2860.jpg,110,"[0.006007458083331585, 0.02774239517748356, 0....",25
23,/train_images/24.jpg,1,"[0.00618132296949625, 0.02886546589434147, 0.0...",25
3071,/train_images/3072.jpg,122,"[0.007186871953308582, 0.028653042390942574, 0...",25


In [94]:
accuracy = accuracy_score(val_df['label'], val_df['predicted_label'])
print(accuracy)

0.008905852417302799


Run the model on test data

In [166]:
test_df = pd.read_csv("../data/test_images_path.csv")

In [167]:
test_dataset = BirdDataset(test_df, transform=test_tfms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [168]:
net.eval()

Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=44944, out_features=400, bias=True)
  (fc2): Linear(in_features=400, out_features=300, bias=True)
  (fc3): Linear(in_features=300, out_features=200, bias=True)
)

In [169]:
all_ids = test_df["id"].tolist()
all_preds = []

In [170]:
with torch.no_grad():
    for inputs, _ in test_loader:
        outputs = net(inputs)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.numpy())

In [171]:
predicted_labels = [idx_to_class[i] for i in all_preds]

In [172]:
output_df = pd.DataFrame({
    "id": all_ids,
    "label": predicted_labels
})

output_df.to_csv("test_predictions.csv", index=False)
print("Saved test_predictions.csv!")

Saved test_predictions.csv!


In [53]:
len(train_df)

3926